# Variational Quantum Eigensolver (VQE)

## **VQE** (Peruzzo et al. 2014) is the flagship near-term quantum algorithm. It finds an approximate ground-state energy of a Hamiltonian $H$ by *variationally* optimising a parameterised quantum circuit. Designed for noisy, shallow devices, it has become the de-facto algorithm for quantum chemistry on today's hardware.

# 1. The variational principle

## For any state $|\psi(\vec\theta)\rangle$,

$$ \Large  \langle\psi(\vec\theta) | H | \psi(\vec\theta)\rangle \;\geq\; E_0, $$

## where $E_0$ is the ground state energy of $H$. Equality holds iff $|\psi(\vec\theta)\rangle$ is a ground state. So minimising $\langle H \rangle_{\vec\theta}$ over the parameters of a quantum circuit gives an upper bound on $E_0$ — and a corresponding approximate ground state.

# 2. The algorithm

## 1. Choose an *ansatz* — a parameterised circuit $U(\vec\theta)$ — and an initial parameter vector $\vec\theta_0$.
## 2. Prepare $|\psi(\vec\theta)\rangle = U(\vec\theta)|0\dots 0\rangle$.
## 3. **Measure** $\langle\psi|H|\psi\rangle$ by expressing $H$ as a sum of Paulis and sampling each term.
## 4. Feed the energy to a **classical optimiser** (gradient descent, COBYLA, SPSA, …) that updates $\vec\theta$.
## 5. Repeat steps 2-4 until convergence.

## Step 3 is where most of the quantum cost lives. Step 4 is purely classical.

In [ ]:
# Conceptual VQE on H = Z_0 Z_1 + X_0 + X_1 using a tiny ansatz
import numpy as np
from scipy.optimize import minimize

# Hamiltonian as a 4x4 matrix
I = np.eye(2); X = np.array([[0,1],[1,0]]); Z = np.array([[1,0],[0,-1]])
H = np.kron(Z, Z) + np.kron(X, I) + np.kron(I, X)

def ansatz(theta):
    # 2-qubit hardware-efficient ansatz: Ry on each + CNOT + Ry on each
    a, b, c, d = theta
    Ry = lambda t: np.array([[np.cos(t/2), -np.sin(t/2)],[np.sin(t/2), np.cos(t/2)]])
    CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]])
    return np.kron(Ry(c), Ry(d)) @ CNOT @ np.kron(Ry(a), Ry(b))

def energy(theta):
    psi = ansatz(theta) @ np.array([1,0,0,0])
    return (psi.conj() @ H @ psi).real

res = minimize(energy, x0=np.random.uniform(0, 2*np.pi, 4), method='COBYLA')
true_E0 = float(np.linalg.eigvalsh(H).min())
print(f'VQE estimate : {res.fun:.6f}')
print(f'True E_0     : {true_E0:.6f}')

# 3. Choosing an ansatz

## - **Hardware-efficient ansatz**: alternating layers of single-qubit rotations and entangling gates from the device's   native gate set. Easy to run, hard to analyse — prone to **barren plateaus** (vanishing gradients in deep circuits).
## - **UCCSD** (unitary coupled cluster, singles & doubles): physically motivated for chemistry, expensive on hardware.
## - **Adaptive ansatzes** (ADAPT-VQE, etc.): grow the circuit greedily based on operator selection.

## The ansatz is the big design choice in VQE and is still an active research area.

# 4. Where VQE shines (and struggles)

## **Shines on**:

## - **Small molecules** with a few qubits — e.g. H$_2$, LiH, BeH$_2$ ground states have been demonstrated on real hardware.
## - Problems where the ansatz captures the relevant physics with few parameters.

## **Struggles with**:

## - Barren plateaus: gradients exponentially small for generic deep ansatzes.
## - Measurement overhead: each evaluation of $\langle H \rangle$ needs many shots, multiplied by many Pauli terms.
## - Optimiser noise from finite shots and hardware noise.
## - Provable advantage over good classical methods (DMRG, coupled cluster) is unclear.

# Recap

## - Variational ground-state search via a parameterised quantum circuit.
## - Hybrid loop: quantum measures $\langle H \rangle$, classical optimiser updates parameters.
## - Designed for noisy, near-term hardware.
## - Open challenges: ansatz design, barren plateaus, measurement cost.